# ConvNeXt Fine-Tuning Pipeline for WFD-2020 Plant Disease Dataset

# 1. Install Dependencies

In [ ]:
!pip install timm albumentations opencv-python

---

# 2. Imports

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm

from sklearn.metrics import f1_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

# 3. Configuration

In [ ]:
class CFG:
    # Paths
    image_dir = '/content/drive/MyDrive/wfd_dataset/'
    train_csv = '/content/drive/MyDrive/data_train.csv'
    valid_csv = '/content/drive/MyDrive/data_valid.csv'
    test_csv = '/content/drive/MyDrive/data_test.csv'

    # Model
    model_name = 'convnext_small.fb_in22k_ft_in1k'

    # Training
    image_size = 224
    batch_size = 8
    epochs = 20
    lr = 2e-5
    weight_decay = 0.05

    # Device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Labels
    labels = [
        'leaf_rust',
        'stem_rust',
        'yellow_rust',
        'powdery_mildew',
        'septoria',
        'healthy',
        'seedlings'
    ]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 4. Dataset

In [ ]:
class WFDDataset(Dataset):
    def __init__(self, df, transforms=None):
        # Updated column name from 'image_id' to 'img' based on CSV inspection
        self.image_ids = df['img'].values
        self.labels = df[CFG.labels].values
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = os.path.join(CFG.image_dir, image_id)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.transforms:
            image = self.transforms(image=image)['image']

        return image, label

# 5. Augmentations

In [ ]:
def get_train_transforms(image_size):
    return A.Compose([
        # Updated to pass size as a tuple to satisfy Albumentations v2.0+
        A.RandomResizedCrop(size=(image_size, image_size), p=1.0),
        A.Transpose(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(p=0.5),
        A.HueSaturationValue(hue_shift_limit=0.2, sat_shift_limit=0.2, val_shift_limit=0.2, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=(-0.1, 0.1), contrast_limit=(-0.1, 0.1), p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0, p=1.0),
        ToTensorV2(p=1.0),
    ], p=1.0)

def get_valid_transforms(image_size):
    return A.Compose([
        A.Resize(height=image_size, width=image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0, p=1.0),
        ToTensorV2(p=1.0),
    ], p=1.0)

# 6. Model

In [ ]:
class ConvNeXt(nn.Module):
    def __init__(self, model_name=CFG.model_name, num_classes=len(CFG.labels), pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained)
        self.model.head.fc = nn.Linear(self.model.head.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

---

# 7. Training Setup

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0
    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1

---

# 8. Main

In [ ]:
def main():
    # Load Data
    train_df = pd.read_csv(CFG.train_csv)
    valid_df = pd.read_csv(CFG.valid_csv)

    # Debug: Print column names to identify the correct ID column
    print(f"Train CSV columns: {train_df.columns.tolist()}")
    print(f"Valid CSV columns: {valid_df.columns.tolist()}")

    # Datasets and DataLoaders
    # Note: If 'image_id' is missing, check the printed columns above and update WFDDataset accordingly.
    train_dataset = WFDDataset(train_df, transforms=get_train_transforms(CFG.image_size))
    valid_dataset = WFDDataset(valid_df, transforms=get_valid_transforms(CFG.image_size))

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=os.cpu_count(),
        pin_memory=True
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=os.cpu_count(),
        pin_memory=True
    )

    # Model, Optimizer, Scheduler, Scaler
    model = ConvNeXt(CFG.model_name, num_classes=len(CFG.labels)).to(CFG.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)

    best_f1 = -1

    for epoch in range(CFG.epochs):
        print(f"Epoch {epoch+1}/{CFG.epochs}")
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, CFG.device)
        valid_loss, valid_f1 = validate_one_epoch(model, valid_loader, criterion, CFG.device)
        scheduler.step()

        print(f"Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}, Valid F1: {valid_f1:.4f}")

        if valid_f1 > best_f1:
            best_f1 = valid_f1
            torch.save(model.state_dict(), 'best_convnext_small.pth')
            print("Saved best model")

    print(f"Best F1 Score: {best_f1:.4f}")

if __name__ == '__main__':
    main()

Train CSV columns: ['img', 'healthy', 'leaf_rust', 'powdery_mildew', 'seedlings', 'septoria', 'stem_rust', 'yellow_rust']
Valid CSV columns: ['img', 'healthy', 'leaf_rust', 'powdery_mildew', 'seedlings', 'septoria', 'stem_rust', 'yellow_rust']


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

/tmp/ipykernel_8255/1025729920.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/20


Training:   0%|          | 0/182 [00:00<?, ?it/s]/tmp/ipykernel_8255/3447935609.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training:  49%|████▉     | 89/182 [09:52<08:19,  5.37s/it]

---

# 9. Inference Example

In [ ]:
test_df = pd.read_csv(CFG.test_csv)
valid_dataset = WFDDataset(test_df, transforms=get_valid_transforms(CFG.image_size))

model = ConvNeXt(CFG.model_name, num_classes=len(CFG.labels)).to(CFG.device)
model.load_state_dict(torch.load('best_convnext_small.pth'))
model.eval()

image, label = valid_dataset[0]

with torch.no_grad():
    image = image.unsqueeze(0).to(CFG.device)

    outputs = model(image)
    probs = torch.sigmoid(outputs)

print(probs)

---

# Important Notes

## Why BCEWithLogitsLoss?

This dataset is multi-label.
A single plant image may contain multiple diseases simultaneously.

So:
- DO NOT use CrossEntropyLoss.
- DO NOT use softmax.

Use:
- BCEWithLogitsLoss
- sigmoid activation during inference.

---

## Recommended Improvements

After your baseline works, try:

1. Test-Time Augmentation (TTA)
2. Exponential Moving Average (EMA)
3. 5-Fold Cross Validation
4. ConvNeXt-Tiny vs Small comparison
5. Progressive resizing:
   - 224 → 384
6. Grad-CAM explainability
7. DINOv2 feature extraction comparison

---